# 漫画图像转视频配音系统

本notebook实现了一个完整的流程：
1. 使用Bedrock Nova多模态模型提取漫画图像关键内容
2. 调用ComfyUI接口进行图生视频
3. 通过GPT-SoVITS接口生成语音
4. 使用MoviePy为视频添加字幕
5. 合并视频、音频和字幕

## 特点
- 🖼️ **批量处理**: 支持目录下多张漫画图像的批量处理
- 🎬 **图生视频**: 使用ComfyUI生成动态视频
- 🔊 **语音合成**: 使用GPT-SoVITS生成高质量语音
- 📝 **智能字幕**: 自动为视频添加基于分析内容的字幕
- 🎞️ **视频合成**: 自动合并多段视频、音频和字幕

## 1. 环境设置和导入

In [1]:
import base64
import boto3
import json
import time
import os
import requests
import urllib.request
import urllib.parse
import uuid
import subprocess
import io
import wave
from datetime import datetime
from typing import Dict, List, Tuple
from pathlib import Path
import numpy as np
from pydub import AudioSegment
import random


# 创建必要的目录
os.makedirs('input_images', exist_ok=True)
os.makedirs('output_videos', exist_ok=True)
os.makedirs('output_audio', exist_ok=True)
os.makedirs('final_videos', exist_ok=True)
os.makedirs('temp', exist_ok=True)

print("✅ 环境设置完成")
print("✅ 目录结构创建完成")

✅ 环境设置完成
✅ 目录结构创建完成


## 1.5. 安装视频字幕依赖

In [2]:
# 安装MoviePy用于视频字幕处理
# 如果还没有安装，请运行以下命令：
# !pip install moviepy

# 验证MoviePy是否正确安装
try:
    from moviepy import VideoFileClip, CompositeVideoClip
    from moviepy import TextClip
    #from moviepy.editor import VideoFileClip, TextClip, CompositeVideoClip
    #from moviepy.video.tools.subtitles import SubtitlesClip
    print("✅ MoviePy 已成功导入，字幕功能可用")
except ImportError as e:
    print(f"❌ MoviePy 导入失败: {e}")
    print("请运行: pip install moviepy")

# 注意：MoviePy 需要 FFmpeg 支持
# 如果遇到 FFmpeg 相关错误，请确保系统已安装 FFmpeg
print("📝 注意：MoviePy 需要 FFmpeg 支持视频处理")
print("如果遇到 FFmpeg 错误，请安装 FFmpeg：")
print("- macOS: brew install ffmpeg")
print("- Ubuntu: sudo apt install ffmpeg")
print("- Windows: 下载 FFmpeg 并添加到 PATH")

✅ MoviePy 已成功导入，字幕功能可用
📝 注意：MoviePy 需要 FFmpeg 支持视频处理
如果遇到 FFmpeg 错误，请安装 FFmpeg：
- macOS: brew install ffmpeg
- Ubuntu: sudo apt install ffmpeg
- Windows: 下载 FFmpeg 并添加到 PATH


## 2. 配置参数

In [3]:
# Bedrock配置
BEDROCK_REGION = "us-west-2"
BEDROCK_MODEL_ID = "us.amazon.nova-pro-v1:0"

# ComfyUI配置 (用户需要后续配置)
COMFYUI_SERVER_URL = "http://ec2-35-84-2-12.us-west-2.compute.amazonaws.com:8188"  # 用户需要填入ComfyUI服务器地址
COMFYUI_WORKFLOW_PATH = "./sample_workflow.json"  # 用户需要填入workflow JSON文件路径

# GPT-SoVITS配置
GPT_SOVITS_ENDPOINT = "gpt-sovits-inference-2025-08-12-07-51-09-161"  # 用户需要填入GPT-SoVITS endpoint名称
REFERENCE_AUDIO_PATH = "s3://sagemaker-us-west-2-687912291502/gpt-sovits/wav/speech_20240425104005663.mp3"  # 用户需要填入参考音频路径
REFERENCE_TEXT = "私はスポーツが好きな女の子で、私は中華料理が大好きで、私は中国へ旅行するのが好きで、特に杭州、成都が好きです"

# 批处理配置
BATCH_SIZE = 6  # 每批处理的图像数量
MAX_IMAGES = 80  # 最大处理图像数量

# 视频生成配置
USE_COMFYUI = True  # True: 使用ComfyUI, False: 使用MoviePy
MOVIEPY_DURATION = 5.0  # MoviePy模式下的视频时长（秒）
MOVIEPY_EFFECT = "random"  # MoviePy动画效果: shake_zoom, pan_zoom, fade_zoom, random

# 注意：JSON结构已简化，不再嵌套，所有字段都是字符串格式
# video_prompt和audio_script是针对整个batch的总体提示词
# selected_index字段用于指定选中的图像序号（从0开始）

print("⚠️ 请确保在运行前配置好以下参数:")
print("- COMFYUI_SERVER_URL")
print("- COMFYUI_WORKFLOW_PATH")
print("- GPT_SOVITS_ENDPOINT")
print("- REFERENCE_AUDIO_PATH")
print("- REFERENCE_TEXT")

⚠️ 请确保在运行前配置好以下参数:
- COMFYUI_SERVER_URL
- COMFYUI_WORKFLOW_PATH
- GPT_SOVITS_ENDPOINT
- REFERENCE_AUDIO_PATH
- REFERENCE_TEXT


## 3. Bedrock Nova 图像内容提取功能

In [4]:
# 创建Bedrock客户端
bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=BEDROCK_REGION,
)

def analyze_comic_image_content(image_path: str, batch_index: int = 0) -> Dict:
    """
    使用Bedrock Nova模型分析漫画图像内容，提取关键信息用于视频生成
    
    Args:
        image_path: 图像文件路径
        batch_index: 批次索引
    
    Returns:
        包含分析结果的字典
    """
    start_time = time.time()
    
    try:
        # 读取图像文件并编码为Base64
        with open(image_path, "rb") as image_file:
            binary_data = image_file.read()
            base_64_encoded_data = base64.b64encode(binary_data)
            base64_string = base_64_encoded_data.decode("utf-8")
        
        # 漫画内容分析提示词
        custom_prompt = """请仔细分析这张漫画图像，并以JSON格式返回以下信息（使用中文回答）：
{
  "scene_description": "场景的详细描述，包括背景、环境、氛围",
  "characters": "人物描述，包括外观、表情、动作、服装",
  "dialogue_text": "图中的对话文字或旁白文字（如果有的话）",
  "story_content": "这一格漫画想要表达的故事内容或情节",
  "visual_style": "画面风格描述，如色彩、线条、构图等",
  "emotion_tone": "整体情感基调（如欢快、紧张、温馨、悲伤等）",
  "video_prompt": "适合用于图生视频的英文提示词，描述如何让这个画面动起来",
  "audio_script": "适合配音的文本内容，可以是对话、旁白或场景描述",
  "key_elements": "画面中的关键元素列表"
}"""
        
        # 定义系统提示
        system_list = [
            {
                "text": "你是一个专业的漫画分析师和视频制作专家，擅长从漫画图像中提取关键信息并转化为视频制作素材。请客观、详细地分析漫画内容。"
            }
        ]
        
        # 定义用户消息
        message_list = [
            {
                "role": "user",
                "content": [
                    {
                        "image": {
                            "format": "png",
                            "source": {"bytes": base64_string},
                        }
                    },
                    {
                        "text": custom_prompt
                    },
                ],
            }
        ]
        
        # 配置推理参数
        inf_params = {"maxTokens": 8000, "topP": 0.1, "topK": 20, "temperature": 0.3}
        
        native_request = {
            "schemaVersion": "messages-v1",
            "messages": message_list,
            "system": system_list,
            "inferenceConfig": inf_params,
        }
        
        # 调用Bedrock API
        api_start_time = time.time()
        response = bedrock_client.invoke_model(modelId=BEDROCK_MODEL_ID, body=json.dumps(native_request))
        model_response = json.loads(response["body"].read())
        api_end_time = time.time()
        
        # 计算时延
        total_latency = api_end_time - start_time
        api_latency = api_end_time - api_start_time
        
        # 提取响应内容
        content_text = model_response["output"]["message"]["content"][0]["text"]
        
        # 尝试解析JSON响应
        try:
            parsed_content = json.loads(content_text)
        except json.JSONDecodeError:
            # 如果不是有效JSON，创建结构化响应
            parsed_content = {
                "scene_description": content_text,
                "characters": "未识别",
                "dialogue_text": "",
                "story_content": content_text,
                "visual_style": "未识别",
                "emotion_tone": "未识别",
                "video_prompt": "animate the comic scene",
                "audio_script": content_text[:200],
                "key_elements": [],
                "raw_response": content_text
            }
        
        return {
            'success': True,
            'file_path': image_path,
            'batch_index': batch_index,
            'analysis_result': parsed_content,
            'raw_response': content_text,
            'model_id': BEDROCK_MODEL_ID,
            'usage': model_response.get('usage', {}),
            'latency': {
                'total_ms': round(total_latency * 1000, 2),
                'api_ms': round(api_latency * 1000, 2)
            },
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        return {
            'success': False,
            'file_path': image_path,
            'batch_index': batch_index,
            'error': str(e),
            'latency': {
                'total_ms': round((time.time() - start_time) * 1000, 2),
                'api_ms': 0
            },
            'timestamp': datetime.now().isoformat()
        }

print("✅ Bedrock Nova 图像分析功能已定义")

✅ Bedrock Nova 图像分析功能已定义


## 4. 批量图像处理功能

In [32]:
def get_comic_images_from_directory(directory_path: str) -> List[str]:
    """
    从目录中获取所有支持的图像文件
    
    Args:
        directory_path: 目录路径
    
    Returns:
        图像文件路径列表
    """
    supported_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}
    
    image_files = []
    
    if not os.path.exists(directory_path):
        print(f"❌ 目录不存在: {directory_path}")
        return image_files
    
    print(f"📂 扫描目录: {directory_path}")
    
    for file in os.listdir(directory_path):
        file_path = os.path.join(directory_path, file)
        if os.path.isfile(file_path):
            file_ext = os.path.splitext(file)[1].lower()
            if file_ext in supported_extensions:
                image_files.append(file_path)
                print(f"  ✅ 找到图像: {file}")
    
    print(f"📊 总共找到 {len(image_files)} 个图像文件")
    return sorted(image_files)

def analyze_comic_images_batch(image_paths: List[str], batch_index: int = 0) -> Dict:
    """
    使用Bedrock Nova模型批量分析多张漫画图像内容，并选择最佳图像用于视频生成
    
    Args:
        image_paths: 图像文件路径列表
        batch_index: 批次索引
    
    Returns:
        包含分析结果和选中图像的字典
    """
    start_time = time.time()
    
    try:
        # 读取所有图像文件并编码为Base64
        image_contents = []
        for i, image_path in enumerate(image_paths):
            with open(image_path, "rb") as image_file:
                binary_data = image_file.read()
                base_64_encoded_data = base64.b64encode(binary_data)
                base64_string = base_64_encoded_data.decode("utf-8")
                
                # 添加图像到内容列表
                image_contents.append({
                    "image": {
                        "format": "png",
                        "source": {"bytes": base64_string},
                    }
                })
        
        # 漫画内容分析提示词（针对多图像）- 简化输出结构
        custom_prompt = f"""请仔细分析这{len(image_paths)}张漫画图像，并返回以下信息（使用中文回答）：

{{
  "overall_analysis": {{
    "story_flow": "整体故事流程和连贯性分析",
    "main_theme": "主要主题和情节发展",
    "character_development": "人物发展和关系变化"
  }},
  "selected_index": "选择为视频输入的图像序号（从0开始，对应提供的图像列表）",
  "video_prompt": "基于整个批次图像的完整视频生成的中文提示词，描述如何让选中的画面动起来",
  "combined_audio_script": "结合整个批次所有图像信息生成的完整配音文本,80字以内"
}}

请分析所有图像的整体故事内容，选择最适合视频生成的一张图像，并提供整个批次的视频生成提示词和完整配音文本。"""
        
        # 定义系统提示
        system_list = [
            {
                "text": "你是一个专业的漫画分析师和视频制作专家，擅长客观、详细地分析漫画内容，从多张漫画图像中提取关键信息、分析故事连贯性。"
            }
        ]
        
        # 构建用户消息内容（包含所有图像和文本）
        content_list = []
        
        # 添加所有图像
        for image_content in image_contents:
            content_list.append(image_content)
        
        # 添加分析提示
        content_list.append({
            "text": custom_prompt
        })
        
        # 定义用户消息
        message_list = [
            {
                "role": "user",
                "content": content_list
            }
        ]
        
        # 配置推理参数（增加token数量以支持多图像分析）
        inf_params = {"maxTokens": 8000, "topP": 0.1, "topK": 20, "temperature": 0.3}
        
        native_request = {
            "schemaVersion": "messages-v1",
            "messages": message_list,
            "system": system_list,
            "inferenceConfig": inf_params,
        }
        
        # 调用Bedrock API
        api_start_time = time.time()
        response = bedrock_client.invoke_model(
            modelId=BEDROCK_MODEL_ID, 
            body=json.dumps(native_request)
        )
        model_response = json.loads(response["body"].read())
        api_end_time = time.time()
        
        # 计算时延
        total_latency = api_end_time - start_time
        api_latency = api_end_time - api_start_time
        
        # 提取响应内容
        content_text = model_response["output"]["message"]["content"][0]["text"]
        
        # 尝试解析JSON响应
        try:
            parsed_content = json.loads(content_text)
            
            # 验证响应结构
            if 'selected_index' in parsed_content:
                selected_index = parsed_content['selected_index']
                # 验证索引是否有效
                if isinstance(selected_index, int) and 0 <= selected_index < len(image_paths):
                    selected_image_path = image_paths[selected_index]
                else:
                    # 如果索引无效，选择第一张图像
                    selected_index = 0
                    selected_image_path = image_paths[0]
                    print(f"⚠️ 选择的图像索引无效，使用第一张图像")
            else:
                # 如果没有选择信息，默认选择第一张
                selected_index = 0
                selected_image_path = image_paths[0]
                parsed_content['selected_index'] = selected_index
                print(f"⚠️ 未找到图像选择信息，使用第一张图像")
                
        except json.JSONDecodeError:
            # 如果不是有效JSON，创建默认结构化响应
            selected_index = 0
            selected_image_path = image_paths[0]
            parsed_content = {
                "overall_analysis": {
                    "story_flow": "解析失败",
                    "main_theme": content_text[:100],
                    "character_development": "未识别"
                },
                "selected_index": selected_index,
                "video_prompt": "animate the comic scene",
                "combined_audio_script": content_text[:300],
                "raw_response": content_text
            }
        
        return {
            'success': True,
            'image_paths': image_paths,
            'selected_image_path': selected_image_path,
            'selected_image_index': selected_index,
            'batch_index': batch_index,
            'analysis_result': parsed_content,
            'raw_response': content_text,
            'model_id': BEDROCK_MODEL_ID,
            'usage': model_response.get('usage', {}),
            'latency': {
                'total_ms': round(total_latency * 1000, 2),
                'api_ms': round(api_latency * 1000, 2)
            },
            'timestamp': datetime.now().isoformat()
        }
        
    except Exception as e:
        return {
            'success': False,
            'image_paths': image_paths,
            'selected_image_path': image_paths[0] if image_paths else None,
            'selected_image_index': 0,
            'batch_index': batch_index,
            'error': str(e),
            'latency': {
                'total_ms': round((time.time() - start_time) * 1000, 2),
                'api_ms': 0
            },
            'timestamp': datetime.now().isoformat()
        }

def batch_analyze_comic_images(directory_path: str, max_files: int = None) -> List[Dict]:
    """
    批量分析目录中的漫画图像（使用多图像API）
    
    Args:
        directory_path: 目录路径
        max_files: 最大处理文件数
    
    Returns:
        分析结果列表（转换为单图像格式以保持兼容性）
    """
    # 获取目录中的所有图像文件
    image_files = get_comic_images_from_directory(directory_path)
    
    if not image_files:
        print("❌ 目录中没有找到支持的图像文件")
        return []
    
    # 限制处理文件数量
    if max_files and len(image_files) > max_files:
        print(f"⚠️ 文件数量超过限制，只处理前 {max_files} 个文件")
        image_files = image_files[:max_files]
    
    results = []
    selected_images_info = []  # 记录每批次选中的图像信息
    total_start_time = time.time()
    
    print(f"\n🚀 开始批量分析 {len(image_files)} 个图像（多图像模式）...\n")
    
    # 按批次处理（每批次包含多张图像）
    for batch_start in range(0, len(image_files), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, len(image_files))
        batch_files = image_files[batch_start:batch_end]
        
        print(f"📦 处理批次 {batch_start//BATCH_SIZE + 1}: 图像 {batch_start+1}-{batch_end}")
        print(f"📁 批次包含图像:")
        for i, file_path in enumerate(batch_files):
            relative_path = os.path.relpath(file_path, directory_path)
            print(f"  {i+1}. {relative_path}")
        
        try:
            # 调用多图像分析API
            result = analyze_comic_images_batch(batch_files, batch_start//BATCH_SIZE)
            
            if result.get('success'):
                analysis = result['analysis_result']

                selected_path = result.get('selected_image_path', '')
                selected_index = result.get('selected_image_index', 0)
                
                print(f"  ✅ 批次分析完成")
                print(f"  🔑 选中图像: 第{selected_index + 1}张 - {os.path.basename(selected_path)}")
                
                # 显示整体分析信息
                if 'overall_analysis' in analysis:
                    overall = analysis['overall_analysis']
                    if 'main_theme' in overall:
                        print(f"  🎭 主要主题: {overall['main_theme']}")
                    
                # 显示视频提示词
                if 'video_prompt' in analysis:
                    video_prompt = analysis['video_prompt']
                    print(f"  🎬 视频提示: {video_prompt}")
                    
                # 显示配音文本
                if 'combined_audio_script' in analysis:
                    audio_script = analysis['combined_audio_script']
                    print(f"  🎙️ 配音文本: {audio_script[:100]}{'...' if len(audio_script) > 100 else ''}")
                
                # 直接使用简化的分析结果（新的JSON结构已经是扁平化的）
                selected_analysis = analysis.copy()
                
                # 创建兼容格式的结果
                converted_result = {
                    'success': True,
                    'file_path': selected_path,
                    'batch_index': batch_start//BATCH_SIZE,
                    'analysis_result': selected_analysis,
                    'raw_response': result.get('raw_response', ''),
                    'model_id': result.get('model_id', ''),
                    'usage': result.get('usage', {}),
                    'latency': result.get('latency', {}),
                    'timestamp': result.get('timestamp', ''),
                    # 新增字段
                    'is_selected_from_batch': True,
                    'batch_image_paths': batch_files,
                    'batch_analysis': analysis
                }
                
                results.append(converted_result)
                
                # 记录选中的图像信息
                selected_images_info.append({
                    'batch_index': batch_start//BATCH_SIZE,
                    'selected_image_path': selected_path,
                    'selected_image_index': selected_index,
                    'batch_files': batch_files,
                    'analysis': selected_analysis
                })
            else:
                print(f"  ❌ 批次分析失败: {result.get('error', 'Unknown error')}")
                # 处理失败的批次
                converted_result = {
                    'success': False,
                    'file_path': batch_files[0] if batch_files else None,
                    'batch_index': batch_start//BATCH_SIZE,
                    'error': result.get('error', 'Unknown error'),
                    'latency': result.get('latency', {}),
                    'timestamp': result.get('timestamp', ''),
                    'is_selected_from_batch': False,
                    'batch_image_paths': batch_files
                }
                results.append(converted_result)
            
            print(f"  ⏱️ 耗时: {result.get('latency', {}).get('total_ms', 0):.2f}ms\n")
            
        except Exception as e:
            print(f"❌ 处理批次失败: {str(e)}")
            converted_result = {
                'success': False,
                'file_path': batch_files[0] if batch_files else None,
                'batch_index': batch_start//BATCH_SIZE,
                'error': str(e),
                'is_selected_from_batch': False,
                'batch_image_paths': batch_files
            }
            results.append(converted_result)
        
        # 避免API限流
        if batch_end < len(image_files):
            print("⏱️ 批次间休息 20 秒...")
            time.sleep(20)
    
    total_time = time.time() - total_start_time
    
    # 打印批量处理统计
    successful = [r for r in results if r.get('success', False)]
    failed = [r for r in results if not r.get('success', False)]
    
    print("📊 批量处理统计:")
    print(f"  • 处理目录: {directory_path}")
    print(f"  • 总图像数: {len(image_files)}")
    print(f"  • 处理批次: {len(results)}")
    print(f"  • 成功批次: {len(successful)}")
    print(f"  • 失败批次: {len(failed)}")
    print(f"  • 选中图像: {len(selected_images_info)}")
    print(f"  • 总耗时: {total_time:.2f}秒")
    
    if successful:
        avg_latency = sum(r.get('latency', {}).get('total_ms', 0) for r in successful) / len(successful)
        print(f"  • 平均延迟: {avg_latency:.2f}ms")
    
    # 保存选中图像信息
    if selected_images_info:
        selected_images_file = os.path.join('temp', 'selected_images_info.json')
        with open(selected_images_file, 'w', encoding='utf-8') as f:
            json.dump(selected_images_info, f, ensure_ascii=False, indent=2)
        print(f"  • 选中图像信息已保存: {selected_images_file}")
    
    return results

print("✅ 批量图像处理功能已定义")

✅ 批量图像处理功能已定义


## 5. 优化提示词和脚本

In [33]:
def optimize_prompts_and_scripts_with_claude(selected_images_file: str = 'temp/selected_images_info.json') -> bool:
    """
    使用Bedrock Claude Sonnet 3.7模型优化video_prompt和combined_audio_script
    
    Args:
        selected_images_file: 选中图像信息文件路径
    
    Returns:
        是否成功优化
    """
    if not os.path.exists(selected_images_file):
        print(f"❌ 文件不存在: {selected_images_file}")
        return False
    
    try:
        # 读取选中图像信息
        with open(selected_images_file, 'r', encoding='utf-8') as f:
            selected_images_info = json.load(f)
        
        print(f"📖 读取到 {len(selected_images_info)} 个批次的信息")
        
        # 创建Claude客户端
        claude_client = boto3.client(
            "bedrock-runtime",
            region_name="us-west-2"
        )
        
        # 优化后的数据
        optimized_data = []
        
        for i, batch_info in enumerate(selected_images_info):
            print(f"\n🔧 优化批次 {i+1}/{len(selected_images_info)}")
            
            # 获取当前批次的信息
            current_analysis = batch_info.get('analysis', {})
            current_video_prompt = current_analysis.get('video_prompt', '')
            current_audio_script = current_analysis.get('combined_audio_script', '')
            
            # 构建之前批次的上下文
            previous_context = ""
            if i > 0:
                previous_batches = []
                for j in range(i):
                    prev_batch = selected_images_info[j]
                    prev_analysis = prev_batch.get('analysis', {})
                    prev_script = prev_analysis.get('combined_audio_script', '')
                    if prev_script:
                        previous_batches.append(f"批次{j+1}: {prev_script}")
                
                if previous_batches:
                    previous_context = "\n".join(previous_batches)
            
            # 构建优化提示词
            optimization_prompt = f"""请帮我优化以下漫画视频生成的提示词和配音脚本：

当前批次信息：
- 批次索引：{batch_info.get('batch_index', i)}
- 当前video_prompt：{current_video_prompt}
- 当前combined_audio_script：{current_audio_script}

之前批次的配音脚本上下文：
{previous_context if previous_context else "无（这是第一个批次）"}

优化要求：
1. 优化video_prompt：
   - 使其符合ComfyUI视频生成模型的prompt提示词格式
   - 去掉不相关的描述（如"让第xx张图像动起来"等）
   - 专注于描述具体的视觉效果、动作、镜头运动等
   - 使用英文，符合AI视频生成模型的标准格式

2. 优化combined_audio_script：
   - 确保与之前批次的人物和主体保持一致性
   - 保持故事的连贯性和逻辑性
   - 适合配音朗读，语言自然流畅
   - 使用中文

请以JSON格式返回优化结果：
{{
  "optimized_video_prompt": "优化后的英文视频生成提示词",
  "optimized_audio_script": "优化后的中文配音脚本，80字以内"
}}"""
            
            # 定义系统提示
            system_list = [
                {
                    "text": "你是一个专业的视频制作和脚本优化专家，擅长优化AI视频生成提示词和配音脚本，确保内容的连贯性和专业性。"
                }
            ]
            
            # 定义用户消息和assistant prefill
            message_list = [
                {
                    "role": "user",
                    "content": [
                        {
                            "text": optimization_prompt
                        }
                    ]
                },
                {
                    "role": "assistant",
                    "content": [
                        {
                            "text": "{"
                        }
                    ]
                }
            ]
            
            # 配置推理参数
            inf_params = {"maxTokens": 4000, "topP": 0.1, "topK": 20, "temperature": 0.3}
            
            native_request = {
                "schemaVersion": "messages-v1",
                "messages": message_list,
                "system": system_list,
                "inferenceConfig": inf_params,
            }
            
            try:
                # 调用Claude Sonnet 3.7
                response = claude_client.invoke_model(
                    #modelId="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
                    modelId=BEDROCK_MODEL_ID,
                    body=json.dumps(native_request)
                )
                
                model_response = json.loads(response["body"].read())
                content_text = model_response["output"]["message"]["content"][0]["text"]
                
                # 尝试解析JSON响应
                try:
                    # 由于使用了assistant prefill "{"，需要补全JSON格式
                    if not content_text.strip().startswith('{'):
                        content_text = '{' + content_text
                    optimization_result = json.loads(content_text)
                    
                    # 更新批次信息
                    optimized_batch = batch_info.copy()
                    optimized_batch['analysis']['video_prompt'] = optimization_result.get('optimized_video_prompt', current_video_prompt)
                    optimized_batch['analysis']['combined_audio_script'] = optimization_result.get('optimized_audio_script', current_audio_script)
                    
                    optimized_data.append(optimized_batch)
                    
                    print(f"  ✅ 批次 {i+1} 优化完成")
                    print(f"  🎬 优化后video_prompt: {optimization_result.get('optimized_video_prompt', '')[:100]}...")
                    print(f"  🎙️ 优化后audio_script: {optimization_result.get('optimized_audio_script', '')[:100]}...")
                    
                except json.JSONDecodeError:
                    print(f"  ⚠️ 批次 {i+1} JSON解析失败，使用原始数据")
                    optimized_data.append(batch_info)
                
            except Exception as e:
                print(f"  ❌ 批次 {i+1} 优化失败: {str(e)}")
                optimized_data.append(batch_info)
            
            # 避免API限流
            if i < len(selected_images_info) - 1:
                time.sleep(30)
        
        # 保存优化后的数据
        with open(selected_images_file, 'w', encoding='utf-8') as f:
            json.dump(optimized_data, f, ensure_ascii=False, indent=2)
        
        print(f"\n✅ 优化完成，已保存到 {selected_images_file}")
        print(f"📊 优化统计: 处理了 {len(optimized_data)} 个批次")
        
        return True
        
    except Exception as e:
        print(f"❌ 优化过程出错: {str(e)}")
        return False

print("✅ Claude Sonnet 3.7 提示词和脚本优化功能已定义")

✅ Claude Sonnet 3.7 提示词和脚本优化功能已定义


## 6. ComfyUI 图生视频功能

In [34]:
import random

def image_to_base64(image_path: str) -> str:
    """
    将图像文件转换为Base64编码
    """
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
            base64_encoded = base64.b64encode(image_data).decode('utf-8')
            return base64_encoded
    except IOError:
        print(f"无法读取文件: {image_path}")
        return None

def queue_prompt(prompt, server_url):
    """
    向ComfyUI服务器提交任务
    """
    client_id = str(uuid.uuid4())
    p = {"prompt": prompt, "client_id": client_id}
    data = json.dumps(p).encode('utf-8')
    url = f"{server_url}/prompt"
    req = urllib.request.Request(url, data=data)
    return json.loads(urllib.request.urlopen(req).read())

def get_video_by_prompt_id(prompt_id, server_url):
    """
    根据prompt_id获取生成的视频
    """
    def get_history(prompt_id):
        with urllib.request.urlopen(f"{server_url}/history/{prompt_id}") as response:
            return json.loads(response.read())
    
    def get_video(filename, subfolder, folder_type):
        data = {"filename": filename, "subfolder": subfolder, "type": folder_type}
        url_values = urllib.parse.urlencode(data)
        with urllib.request.urlopen(f"{server_url}/view?{url_values}") as response:
            return response.read()
    
    output_videos = {}
    while True:
        try:
            history = get_history(prompt_id)[prompt_id]
            for node_id in history['outputs']:
                node_output = history['outputs'][node_id]
                # 视频输出分支
                if 'gifs' in node_output:
                    videos_output = []
                    for video in node_output['gifs']:
                        video_data = get_video(video['filename'], video['subfolder'], video['type'])
                        videos_output.append(video_data)
                    output_videos[node_id] = videos_output
            break
        except Exception as e:
            print(f"等待执行历史: {e}")
            time.sleep(15)
            continue
    
    return output_videos

def generate_video_from_image(image_path: str, video_prompt: str, output_path: str) -> bool:
    """
    使用ComfyUI从图像生成视频
    
    Args:
        image_path: 输入图像路径
        video_prompt: 视频生成提示词
        output_path: 输出视频路径
    
    Returns:
        是否成功生成视频
    """
    if not COMFYUI_SERVER_URL or not COMFYUI_WORKFLOW_PATH:
        print("❌ ComfyUI配置未完成，请设置COMFYUI_SERVER_URL和COMFYUI_WORKFLOW_PATH")
        return False
    
    try:
        # 读取workflow模板
        with open(COMFYUI_WORKFLOW_PATH, 'r') as f:
            workflow = json.load(f)
        
        # 将图像转换为base64
        base64_image = image_to_base64(image_path)
        if not base64_image:
            return False
        
        # 修改workflow中的参数（这里需要根据实际的workflow结构调整）
        # 假设workflow中有图像输入节点和文本提示节点
        # 根据实际workflow调整这些节点ID和参数名
        
        # 示例：设置图像输入（需要根据实际workflow调整）
        if '29' in workflow:  # 替换为实际的节点ID
            workflow['29']['inputs']['image'] = base64_image
        
        # 示例：设置文本提示（需要根据实际workflow调整）
        if '33' in workflow:  # 替换为实际的节点ID
            workflow['33']['inputs']['text'] = video_prompt
        
        if '28' in workflow:
            workflow['33']['inputs']['seed'] = random.randint(0, 999999998)
        
        print(f"🎬 开始生成视频: {os.path.basename(image_path)}")
        print(f"📝 视频提示词: {video_prompt}")
        
        # 提交任务
        result = queue_prompt(workflow, COMFYUI_SERVER_URL)
        prompt_id = result['prompt_id']
        print(f"📋 任务ID: {prompt_id}")
        
        # 获取生成的视频
        output_videos = get_video_by_prompt_id(prompt_id, COMFYUI_SERVER_URL)
        
        if output_videos:
            # 保存第一个生成的视频
            for node_id, videos in output_videos.items():
                if videos:
                    with open(output_path, 'wb') as f:
                        f.write(videos[0])
                    print(f"✅ 视频已保存: {output_path}")
                    return True
        
        print("❌ 未生成视频")
        return False
        
    except Exception as e:
        print(f"❌ 视频生成失败: {str(e)}")
        return False

print("✅ ComfyUI 图生视频功能已定义")
print("⚠️ 注意：需要根据实际的ComfyUI workflow调整节点ID和参数")

✅ ComfyUI 图生视频功能已定义
⚠️ 注意：需要根据实际的ComfyUI workflow调整节点ID和参数


## 6.5. MoviePy 图像动画视频生成功能

In [ ]:
def generate_video_from_image_moviepy(image_path: str, video_prompt: str, output_path: str, duration: float = 5.0, effect: str = "shake_zoom") -> bool:
    """
    使用MoviePy从图像生成动画视频
    
    Args:
        image_path: 输入图像路径
        video_prompt: 视频生成提示词（用于记录，不直接影响生成）
        output_path: 输出视频路径
        duration: 视频时长（秒）
        effect: 动画效果类型
    
    Returns:
        是否成功生成视频
    """
    try:
        from moviepy.editor import ImageClip, CompositeVideoClip
        import numpy as np
        import random
        
        # 如果效果是random，随机选择一个效果
        if effect == "random":
            available_effects = ["shake_zoom", "fade_zoom", "pan_zoom"]
            effect = random.choice(available_effects)
        
        print(f"🎬 使用MoviePy生成动画视频: {os.path.basename(image_path)}")
        print(f"📝 视频提示词: {video_prompt}")
        print(f"⏱️ 时长: {duration}秒, 效果: {effect}")
        
        # 加载图像
        clip = ImageClip(image_path, duration=duration)
        
        # 应用动画效果
        if effect == "shake_zoom":
            # 震动+缩放效果
            def shake_zoom_effect(get_frame, t):
                frame = get_frame(t)
                
                # 计算震动偏移
                shake_intensity = 5
                shake_x = int(shake_intensity * np.sin(t * 20))
                shake_y = int(shake_intensity * np.cos(t * 15))
                
                # 计算缩放因子（从1.0到1.1）
                zoom_factor = 1.0 + 0.1 * (t / duration)
                
                # 应用缩放
                h, w = frame.shape[:2]
                new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
                
                # 简单的缩放（使用重复像素）
                if new_h > h and new_w > w:
                    # 裁剪中心区域
                    start_y = (new_h - h) // 2
                    start_x = (new_w - w) // 2
                    # 这里简化处理，直接返回原frame加震动
                    pass
                
                # 应用震动（通过填充实现）
                if abs(shake_x) < frame.shape[1]//4 and abs(shake_y) < frame.shape[0]//4:
                    # 创建新的frame
                    new_frame = np.zeros_like(frame)
                    
                    # 计算源和目标区域
                    src_y1 = max(0, -shake_y)
                    src_y2 = min(frame.shape[0], frame.shape[0] - shake_y)
                    src_x1 = max(0, -shake_x)
                    src_x2 = min(frame.shape[1], frame.shape[1] - shake_x)
                    
                    dst_y1 = max(0, shake_y)
                    dst_y2 = dst_y1 + (src_y2 - src_y1)
                    dst_x1 = max(0, shake_x)
                    dst_x2 = dst_x1 + (src_x2 - src_x1)
                    
                    new_frame[dst_y1:dst_y2, dst_x1:dst_x2] = frame[src_y1:src_y2, src_x1:src_x2]
                    return new_frame
                
                return frame
            
            clip = clip.fl(shake_zoom_effect)
            
        elif effect == "pan_zoom":
            # 平移+缩放效果
            def pan_zoom_effect(get_frame, t):
                frame = get_frame(t)
                # 简单的缩放效果
                zoom_factor = 1.0 + 0.2 * (t / duration)
                return frame
            
            clip = clip.fl(pan_zoom_effect)
            
        elif effect == "fade_zoom":
            # 淡入淡出+缩放效果
            clip = clip.fadein(0.5).fadeout(0.5)
            
            def zoom_effect(get_frame, t):
                frame = get_frame(t)
                return frame
            
            clip = clip.fl(zoom_effect)
        
        # 设置帧率
        clip = clip.set_fps(16)
        
        # 写入视频文件
        clip.write_videofile(output_path, codec='libx264', audio=False, verbose=False, logger=None)
        
        # 清理资源
        clip.close()
        
        print(f"✅ MoviePy视频已保存: {output_path}")
        return True
        
    except Exception as e:
        print(f"❌ MoviePy视频生成失败: {str(e)}")
        return False

def generate_video_with_option(image_path: str, video_prompt: str, output_path: str, 
                              use_comfyui: bool = True, duration: float = 5.0, effect: str = "shake_zoom") -> bool:
    """
    根据选项使用ComfyUI或MoviePy生成视频
    
    Args:
        image_path: 输入图像路径
        video_prompt: 视频生成提示词
        output_path: 输出视频路径
        use_comfyui: 是否使用ComfyUI（True）或MoviePy（False）
        duration: MoviePy模式下的视频时长
        effect: MoviePy模式下的动画效果
    
    Returns:
        是否成功生成视频
    """
    if use_comfyui and COMFYUI_SERVER_URL and COMFYUI_WORKFLOW_PATH:
        print(f"🎬 使用ComfyUI生成视频")
        return generate_video_from_image(image_path, video_prompt, output_path)
    else:
        if use_comfyui:
            print(f"⚠️ ComfyUI配置不完整，切换到MoviePy模式")
        else:
            print(f"🎬 使用MoviePy生成视频")
        return generate_video_from_image_moviepy(image_path, video_prompt, output_path, duration, effect)

print("✅ MoviePy 图像动画视频生成功能已定义")
print("💡 支持的动画效果: shake_zoom, pan_zoom, fade_zoom")

## 7. GPT-SoVITS 语音生成功能

In [35]:
import boto3
from botocore.config import Config

def upsert(lst, new_dict):
    """
    更新或插入字典到列表中
    """
    for i, item in enumerate(lst):
        if new_dict['index'] == i:
            lst[i] = new_dict
            return lst
    lst.append(new_dict)
    return lst

def invoke_gpt_sovits_endpoint(smr_client, endpoint_name, request):
    """
    调用GPT-SoVITS端点生成语音
    """
    content_type = "application/json"
    payload = json.dumps(request, ensure_ascii=False)

    response_model = smr_client.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        ContentType=content_type,
        Body=payload,
    )

    result = []
    print(f"📡 响应元数据: {response_model['ResponseMetadata']}")
    event_stream = iter(response_model['Body'])
    index = 0
    chunk_bytes = None
    
    try: 
        while True:
            event = next(event_stream)
            eventChunk = event['PayloadPart']['Bytes']
            chunk_dict = {}
            if index == 0:
                print("📦 收到第一个音频块")
                chunk_dict['first_chunk'] = True
                chunk_dict['bytes'] = eventChunk
                chunk_bytes = eventChunk
                chunk_dict['last_chunk'] = False
                chunk_dict['index'] = index
            else:
                chunk_dict['first_chunk'] = False
                chunk_dict['bytes'] = eventChunk
                chunk_bytes = eventChunk
                chunk_dict['last_chunk'] = False
                chunk_dict['index'] = index
            print(f"📦 音频块长度: {len(chunk_dict['bytes'])}")
            result.append(chunk_dict)    
            index += 1
    except StopIteration:
        print("✅ 所有音频块处理完成")
        chunk_dict = {}
        chunk_dict['first_chunk'] = False
        chunk_dict['bytes'] = chunk_bytes
        chunk_dict['last_chunk'] = True
        chunk_dict['index'] = index-1
        result = upsert(result, chunk_dict)
    
    return result

def generate_audio_from_text(output_path: str, prompt_text: str = None) -> bool:
    """
    使用GPT-SoVITS从文本生成语音
    
    Args:
        text: 要转换为语音的文本
        output_path: 输出音频文件路径
        prompt_text: 提示文本（可选）
    
    Returns:
        是否成功生成音频
    """
    if not GPT_SOVITS_ENDPOINT or not REFERENCE_AUDIO_PATH:
        print("❌ GPT-SoVITS配置未完成，请设置GPT_SOVITS_ENDPOINT和REFERENCE_AUDIO_PATH")
        return False
    
    try:
        my_config = Config(
            connect_timeout=10,  # 连接超时，单位为秒
            read_timeout=300      # 读取超时，单位为秒
        )

        runtime_sm_client = boto3.client(service_name="sagemaker-runtime",
                                         region_name="us-east-1",
                                         config=my_config)
        
        
        # 构建请求数据
        data = {
            "text": prompt_text,
            "text_lang": "zh",
            "ref_audio_path": REFERENCE_AUDIO_PATH,
            "prompt_lang": "ja",
            "prompt_text": REFERENCE_TEXT,
            "top_k": 5,
            "top_p": 1.0,
            "temperature": 0.7,
            "text_split_method": "cut3",
            "batch_size": 1,
            "batch_threshold": 0.75,
            "split_bucket": True,
            "speed_factor": 1.0,
            "fragment_interval": 0.3,
            "seed": -1,
            "media_type": "wav",
            "streaming_mode": True,
            "parallel_infer": True,
            "repetition_penalty": 1.35,
            "sample_steps": 32,
            "super_sampling": False
        }
        
        print(f"🔊 开始生成语音")
        print(f"📝 文本内容: {prompt_text[:100] if prompt_text else 'None'}{'...' if prompt_text and len(prompt_text) > 100 else ''}")
        
        # 调用GPT-SoVITS端点
        response = invoke_gpt_sovits_endpoint(runtime_sm_client, GPT_SOVITS_ENDPOINT, data)
        
        # 合并音频数据
        audio_data = b''.join(chunk['bytes'] for chunk in response)
        
        # 保存音频文件
        with open(output_path, 'wb') as f:
            f.write(audio_data)
        
        print(f"✅ 音频已保存: {output_path}")
        return True
        
    except Exception as e:
        print(f"❌ 语音生成失败: {str(e)}")
        return False

print("✅ GPT-SoVITS 语音生成功能已定义")

✅ GPT-SoVITS 语音生成功能已定义


## 8. 视频和音频合并功能

In [36]:
def merge_video_audio(video_path: str, audio_path: str, output_path: str) -> bool:
    """
    合并视频和音频文件
    
    Args:
        video_path: 视频文件路径
        audio_path: 音频文件路径
        output_path: 输出文件路径
    
    Returns:
        是否成功合并
    """
    try:
        cmd = [
            'ffmpeg',
            '-i', video_path,
            '-i', audio_path,
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-strict', 'experimental',
            '-y',  # 覆盖输出文件
            output_path
        ]
        
        print(f"🎞️ 合并视频和音频")
        print(f"📹 视频: {os.path.basename(video_path)}")
        print(f"🔊 音频: {os.path.basename(audio_path)}")
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode == 0:
            print(f"✅ 合并完成: {output_path}")
            return True
        else:
            print(f"❌ 合并失败: {result.stderr}")
            return False
            
    except Exception as e:
        print(f"❌ 合并过程出错: {str(e)}")
        return False

def concatenate_videos(video_paths: List[str], output_path: str) -> bool:
    """
    拼接多个视频文件
    
    Args:
        video_paths: 视频文件路径列表
        output_path: 输出文件路径
    
    Returns:
        是否成功拼接
    """
    if not video_paths:
        print("❌ 没有视频文件需要拼接")
        return False
    
    if len(video_paths) == 1:
        # 只有一个视频，直接复制
        try:
            import shutil
            shutil.copy2(video_paths[0], output_path)
            print(f"✅ 单个视频已复制: {output_path}")
            return True
        except Exception as e:
            print(f"❌ 复制视频失败: {str(e)}")
            return False
    
    try:
        # 创建临时文件列表
        temp_list_file = os.path.join('temp', 'video_list.txt')
        with open(temp_list_file, 'w') as f:
            for video_path in video_paths:
                # 使用绝对路径避免路径问题
                abs_path = os.path.abspath(video_path)
                f.write(f"file '{abs_path}'\n")
        
        cmd = [
            'ffmpeg',
            '-f', 'concat',
            '-safe', '0',
            '-i', temp_list_file,
            '-c', 'copy',
            '-y',  # 覆盖输出文件
            output_path
        ]
        
        print(f"🎬 拼接 {len(video_paths)} 个视频文件")
        for i, path in enumerate(video_paths, 1):
            print(f"  {i}. {os.path.basename(path)}")
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        # 清理临时文件
        if os.path.exists(temp_list_file):
            os.remove(temp_list_file)
        
        if result.returncode == 0:
            print(f"✅ 视频拼接完成: {output_path}")
            return True
        else:
            print(f"❌ 视频拼接失败: {result.stderr}")
            return False
            
    except Exception as e:
        print(f"❌ 拼接过程出错: {str(e)}")
        return False

def get_video_duration(video_path: str) -> float:
    """
    获取视频时长（秒）
    """
    try:
        cmd = [
            'ffprobe',
            '-v', 'quiet',
            '-print_format', 'json',
            '-show_format',
            video_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            info = json.loads(result.stdout)
            duration = float(info['format']['duration'])
            return duration
        else:
            print(f"❌ 获取视频时长失败: {result.stderr}")
            return 0.0
            
    except Exception as e:
        print(f"❌ 获取视频时长出错: {str(e)}")
        return 0.0

print("✅ 视频和音频合并功能已定义")

✅ 视频和音频合并功能已定义


## 8.5. 视频字幕功能

In [41]:
def add_subtitles_to_video(video_path: str, subtitle_text: str, output_path: str, 
                          font_size: int = 10, font_color: str = 'white', 
                          bg_color: str = 'black', position: str = 'bottom') -> bool:
    """
    为视频添加字幕
    Args:
        video_path: 输入视频路径
        subtitle_text: 字幕文本
        output_path: 输出视频路径
        font_size: 字体大小
        font_color: 字体颜色
        bg_color: 背景颜色
        position: 字幕位置 ('bottom', 'top', 'center')   
    Returns:
        是否成功添加字幕
    """
    try:
        # 设置FFmpeg环境变量（修复字幕问题）
        import os
        if 'IMAGEIO_FFMPEG_EXE' not in os.environ:
            # 尝试找到ffmpeg
            import shutil
            ffmpeg_path = shutil.which('ffmpeg')
            if ffmpeg_path:
                os.environ['IMAGEIO_FFMPEG_EXE'] = ffmpeg_path
        
        # 加载视频\n
        video = VideoFileClip(video_path)
        # 计算字幕位置\n
        if position == 'bottom':
            subtitle_position = ('center', video.h - 100)
        elif position == 'top':
            subtitle_position = ('center', 50)
        else:  # center\n
            subtitle_position = ('center', 'center')
        # 创建字幕文本片段\n
        subtitle_clip = TextClip(
            text=subtitle_text if len(subtitle_text) <= 20 else '\n'.join([subtitle_text[i:i+20] for i in range(0, len(subtitle_text), 20)]),
            font_size=font_size,
            color=font_color,
            font='./yahei.ttf',  # 使用系统字体\n
            margin=(20,20),
            text_align='center'
        ).with_position(position).with_duration(video.duration)
        # 合成视频和字幕\n
        final_video = CompositeVideoClip([video, subtitle_clip])
        print(f"📝 添加字幕到视频")
        print(f"📹 输入视频: {os.path.basename(video_path)}")
        print(f"💬 字幕内容: {subtitle_text[:50]}{'...' if len(subtitle_text) > 50 else ''}")
        # 写入输出文件\n
        # 创建临时音频文件路径
        temp_audio_path = os.path.join('temp', f'temp_audio_{uuid.uuid4().hex[:8]}.m4a')
        
        final_video.write_videofile(output_path)
        
        
        # 清理资源\n
        video.close()
        subtitle_clip.close()
        final_video.close()
        
        print(f"✅ 字幕视频已保存: {output_path}")
        return True
    except Exception as e:
        print(f"❌ 添加字幕失败: {str(e)}")
        return False
def merge_video_audio_with_subtitles(video_path: str, audio_path: str, 
                                   subtitle_text: str, output_path: str) -> bool:
    """
    
    合并视频、音频并添加字幕
    
    Args:
        video_path: 视频文件路径\n
        audio_path: 音频文件路径\n
        subtitle_text: 字幕文本\n
        output_path: 输出文件路径\n
    
    Returns:\n
        是否成功处理\n
    """
    try:
        # 临时文件路径\n
        temp_video_with_audio = os.path.join('temp', f'temp_merged_{uuid.uuid4().hex[:8]}.mp4')
        # 第一步：合并视频和音频\n
        if not merge_video_audio(video_path, audio_path, temp_video_with_audio):
            return False
        # 第二步：添加字幕\n
        success = add_subtitles_to_video(temp_video_with_audio, subtitle_text, output_path)
        # 清理临时文件\n
        if os.path.exists(temp_video_with_audio):
            print(f"💾 保留临时文件用于调试: {temp_video_with_audio}")
            #os.remove(temp_video_with_audio)
        return success
    except Exception as e:
        print("❌ 合并视频音频并添加字幕失败: {str(e)}")
        return False
print("✅ 视频字幕功能已定义")

✅ 视频字幕功能已定义


## 9. 主要处理流程

In [42]:
def process_comic_to_video_voice(input_directory: str) -> str:
    """
    完整的漫画转视频配音流程
    
    Args:
        input_directory: 输入漫画图像目录
    
    Returns:
        最终输出视频路径
    """
    print("🚀 开始漫画转视频配音流程")
    print("="*60)
    
    # 步骤1: 批量分析漫画图像
    print("\n📊 步骤1: 分析漫画图像内容")
    analysis_results = batch_analyze_comic_images(input_directory, MAX_IMAGES)
    
    if not analysis_results:
        print("❌ 没有成功分析的图像")
        return None
    
    # 过滤成功的分析结果
    successful_results = [r for r in analysis_results if r.get('success')]
    if not successful_results:
        print("❌ 没有成功分析的图像")
        return None
    
    print(f"✅ 成功分析 {len(successful_results)} 张图像")
    
    # 步骤2: 使用Claude Sonnet 3.7优化提示词和脚本
    print("\n🔧 步骤2: 优化视频提示词和配音脚本")
    if not optimize_prompts_and_scripts_with_claude():
        print("⚠️ 提示词优化失败，继续使用原始提示词")
    else:
        print("✅ 提示词和脚本优化完成")
    
    # 步骤3: 生成视频
    print("\n🎬 步骤3: 生成视频")
    video_files = []
    audio_scripts = []
    
    # 读取优化后的selected_images_info.json文件
    selected_images_file = 'temp/selected_images_info.json'
    optimized_data = []
    
    if os.path.exists(selected_images_file):
        try:
            with open(selected_images_file, 'r', encoding='utf-8') as f:
                optimized_data = json.load(f)
            print(f"📖 读取到优化后的 {len(optimized_data)} 个批次数据")
        except Exception as e:
            print(f"⚠️ 读取优化数据失败: {e}，使用原始数据")
            optimized_data = []
    
    # 如果有优化数据，使用优化数据；否则使用原始数据
    if optimized_data:
        print("✅ 使用优化后的提示词和脚本")
        for i, batch_data in enumerate(optimized_data):
            analysis = batch_data.get('analysis', {})
            image_path = batch_data.get('selected_image_path', '')
            
            # 生成视频文件名
            video_filename = f"video_{i:03d}.mp4"
            video_path = os.path.join('output_videos', video_filename)
            
            # 获取优化后的视频提示词
            video_prompt = analysis.get('video_prompt', 'animate the comic scene')
            
            print(f"\n🎥 生成视频 {i+1}/{len(optimized_data)}")
            print(f"📝 使用优化后的视频提示词: {video_prompt[:100]}...")
            
            if generate_video_with_option(image_path, video_prompt, video_path, USE_COMFYUI, MOVIEPY_DURATION, MOVIEPY_EFFECT):
                video_files.append(video_path)
                # 收集优化后的音频脚本
                audio_script = analysis.get('combined_audio_script', '')
                if audio_script and audio_script.strip():
                    audio_scripts.append(audio_script)
                else:
                    audio_scripts.append(f"第{i+1}个场景")
            else:
                print(f"⚠️ 视频 {i+1} 生成失败，跳过")
    else:
        print("⚠️ 使用原始数据作为备选方案")
        # 使用原始数据作为备选方案
        for i, result in enumerate(successful_results):
            analysis = result['analysis_result']
            image_path = result['file_path']
            
            # 生成视频文件名
            video_filename = f"video_{i:03d}.mp4"
            video_path = os.path.join('output_videos', video_filename)
            
            # 获取视频提示词
            video_prompt = analysis.get('video_prompt', 'animate the comic scene')
            
            print(f"\n🎥 生成视频 {i+1}/{len(successful_results)}")
            if generate_video_with_option(image_path, video_prompt, video_path, USE_COMFYUI, MOVIEPY_DURATION, MOVIEPY_EFFECT):
                video_files.append(video_path)
                # 收集音频脚本（使用新的combined_audio_script字段）
                audio_script = analysis.get('combined_audio_script', '')
                if audio_script and audio_script.strip():
                    audio_scripts.append(audio_script)
                else:
                    audio_scripts.append(f"第{i+1}个场景")
            else:
                print(f"⚠️ 视频 {i+1} 生成失败，跳过")
    
    if not video_files:
        print("❌ 没有成功生成的视频")
        return None
    
    print(f"✅ 成功生成 {len(video_files)} 个视频")
    
    # 步骤4: 生成语音
    print("\n🔊 步骤4: 生成语音")
    audio_files = []
    
    for i, script in enumerate(audio_scripts):
        audio_filename = f"audio_{i:03d}.wav"
        audio_path = os.path.join('output_audio', audio_filename)
        
        print(f"\n🎙️ 生成语音 {i+1}/{len(audio_scripts)}")
        print(f"📝 使用优化后的配音脚本: {script[:100]}...")
        if generate_audio_from_text(output_path=audio_path,prompt_text=script):
            audio_files.append(audio_path)
        else:
            print(f"⚠️ 语音 {i+1} 生成失败，跳过")
            audio_files.append(None)
    
    print(f"✅ 成功生成 {len([a for a in audio_files if a])} 个语音文件")
    
    # 步骤5: 合并视频、音频并添加字幕
    print("\n🎞️ 步骤5: 合并视频、音频并添加字幕")
    final_video_files = []
    
    for i, (video_path, audio_path) in enumerate(zip(video_files, audio_files)):
        final_filename = f"final_{i:03d}.mp4"
        final_path = os.path.join('final_videos', final_filename)
        
        # 获取对应的字幕文本
        subtitle_text = audio_scripts[i] if i < len(audio_scripts) else ""
        
        if audio_path and os.path.exists(audio_path):
            print(f"\n🎬 合并视频音频并添加字幕 {i+1}/{len(video_files)}")
            if merge_video_audio_with_subtitles(video_path, audio_path, subtitle_text, final_path):
                final_video_files.append(final_path)
            else:
                print(f"⚠️ 合并失败，尝试仅添加字幕")
                # 如果合并失败，尝试只添加字幕
                if subtitle_text and add_subtitles_to_video(video_path, subtitle_text, final_path):
                    final_video_files.append(final_path)
                else:
                    print(f"⚠️ 字幕添加也失败，使用原视频")
                    final_video_files.append(video_path)
        else:
            print(f"⚠️ 没有对应音频，仅添加字幕")
            if subtitle_text and add_subtitles_to_video(video_path, subtitle_text, final_path):
                final_video_files.append(final_path)
            else:
                print(f"⚠️ 没有字幕文本，使用原视频")
                final_video_files.append(video_path)
    
    # 步骤6: 拼接所有视频
    print("\n🎬 步骤6: 拼接最终视频")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    final_output = os.path.join('final_videos', f"comic_video_{timestamp}.mp4")
    
    if concatenate_videos(final_video_files, final_output):
        print(f"\n🎉 流程完成！")
        print(f"📁 最终视频: {final_output}")
        
        # 显示统计信息
        total_duration = sum(get_video_duration(vf) for vf in final_video_files if os.path.exists(vf))
        print(f"⏱️ 总时长: {total_duration:.2f} 秒")
        print(f"📊 处理统计:")
        print(f"  • 输入图像: {len(analysis_results)}")
        print(f"  • 生成视频: {len(video_files)}")
        print(f"  • 生成语音: {len([a for a in audio_files if a])}")
        print(f"  • 最终视频位置: final_videos/ 目录")
        
        return final_output
    else:
        print("❌ 最终视频拼接失败")
        return None

def process_selected_images_with_moviepy(selected_images_file: str = 'temp/selected_images_info.json') -> str:
    """
    使用selected_images_info.json文件中的信息，通过MoviePy生成视频
    
    Args:
        selected_images_file: 选中图像信息文件路径
    
    Returns:
        最终输出视频路径
    """
    print("🎬 使用MoviePy处理选中图像生成视频")
    print("="*60)
    
    if not os.path.exists(selected_images_file):
        print(f"❌ 文件不存在: {selected_images_file}")
        return None
    
    try:
        # 读取选中图像信息
        with open(selected_images_file, 'r', encoding='utf-8') as f:
            selected_images_info = json.load(f)
        
        print(f"📖 读取到 {len(selected_images_info)} 个批次的信息")
        
        video_files = []
        audio_scripts = []
        
        # 为每个批次生成视频
        for i, batch_info in enumerate(selected_images_info):
            print(f"\n🎥 处理批次 {i+1}/{len(selected_images_info)}")
            
            # 获取批次信息
            image_path = batch_info.get('selected_image_path', '')
            analysis = batch_info.get('analysis', {})
            video_prompt = analysis.get('video_prompt', 'Two people\'s first encounter and tension')
            narration = analysis.get('combined_audio_script', f'第{i+1}个场景')
            
            # 构建MoviePy配置
            moviepy_config = {
                "image": image_path,
                "duration": 5,
                "narration": narration,
                "effect": "random"
            }
            
            print(f"📁 图像路径: {os.path.basename(image_path)}")
            print(f"⏱️ 时长: {moviepy_config['duration']}秒")
            print(f"🎭 效果: {moviepy_config['effect']}")
            print(f"📝 旁白: {moviepy_config['narration'][:50]}...")
            
            # 生成视频文件名
            video_filename = f"moviepy_video_{i:03d}.mp4"
            video_path = os.path.join('output_videos', video_filename)
            
            # 使用MoviePy生成视频
            if generate_video_from_image_moviepy(
                image_path=moviepy_config['image'],
                video_prompt=video_prompt,
                output_path=video_path,
                duration=moviepy_config['duration'],
                effect=moviepy_config['effect']
            ):
                video_files.append(video_path)
                audio_scripts.append(moviepy_config['narration'])
                print(f"✅ 视频生成成功: {video_filename}")
            else:
                print(f"❌ 视频生成失败: {video_filename}")
        
        if not video_files:
            print("❌ 没有成功生成的视频")
            return None
        
        print(f"\n✅ 成功生成 {len(video_files)} 个视频")
        
        # 生成语音（如果配置了GPT-SoVITS）
        audio_files = []
        if GPT_SOVITS_ENDPOINT and REFERENCE_AUDIO_PATH:
            print("\n🔊 生成语音")
            for i, script in enumerate(audio_scripts):
                audio_filename = f"moviepy_audio_{i:03d}.wav"
                audio_path = os.path.join('output_audio', audio_filename)
                
                print(f"\n🎙️ 生成语音 {i+1}/{len(audio_scripts)}")
                if generate_audio_from_text(output_path=audio_path, prompt_text=script):
                    audio_files.append(audio_path)
                else:
                    audio_files.append(None)
        else:
            print("\n⚠️ 跳过语音生成（GPT-SoVITS未配置）")
            audio_files = [None] * len(video_files)
        
        # 合并视频、音频并添加字幕
        print("\n🎞️ 合并视频、音频并添加字幕")
        final_video_files = []
        
        for i, (video_path, audio_path) in enumerate(zip(video_files, audio_files)):
            final_filename = f"moviepy_final_{i:03d}.mp4"
            final_path = os.path.join('final_videos', final_filename)
            
            subtitle_text = audio_scripts[i] if i < len(audio_scripts) else ""
            
            if audio_path and os.path.exists(audio_path):
                print(f"\n🎬 合并视频音频并添加字幕 {i+1}/{len(video_files)}")
                if merge_video_audio_with_subtitles(video_path, audio_path, subtitle_text, final_path):
                    final_video_files.append(final_path)
                else:
                    print(f"⚠️ 合并失败，尝试仅添加字幕")
                    if subtitle_text and add_subtitles_to_video(video_path, subtitle_text, final_path):
                        final_video_files.append(final_path)
                    else:
                        final_video_files.append(video_path)
            else:
                print(f"⚠️ 没有对应音频，仅添加字幕 {i+1}/{len(video_files)}")
                if subtitle_text and add_subtitles_to_video(video_path, subtitle_text, final_path):
                    final_video_files.append(final_path)
                else:
                    final_video_files.append(video_path)
        
        # 拼接所有视频
        print("\n🎬 拼接最终视频")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_output = os.path.join('final_videos', f"moviepy_comic_video_{timestamp}.mp4")
        
        if concatenate_videos(final_video_files, final_output):
            print(f"\n🎉 MoviePy流程完成！")
            print(f"📁 最终视频: {final_output}")
            
            # 显示统计信息
            total_duration = sum(get_video_duration(vf) for vf in final_video_files if os.path.exists(vf))
            print(f"⏱️ 总时长: {total_duration:.2f} 秒")
            print(f"📊 处理统计:")
            print(f"  • 输入批次: {len(selected_images_info)}")
            print(f"  • 生成视频: {len(video_files)}")
            print(f"  • 生成语音: {len([a for a in audio_files if a])}")
            print(f"  • 最终视频位置: final_videos/ 目录")
            
            return final_output
        else:
            print("❌ 最终视频拼接失败")
            return None
            
    except Exception as e:
        print(f"❌ 处理过程中发生错误: {str(e)}")
        return None

print("✅ 主要处理流程已定义")
print("✅ MoviePy专用处理流程已定义")

✅ 主要处理流程已定义


## 10. 使用示例

In [43]:
# 配置示例（用户需要根据实际情况修改）
print("📋 配置检查")
print("请确保以下配置已正确设置：")
print(f"• ComfyUI服务器: {COMFYUI_SERVER_URL or '未设置'}")
print(f"• ComfyUI工作流: {COMFYUI_WORKFLOW_PATH or '未设置'}")
print(f"• GPT-SoVITS端点: {GPT_SOVITS_ENDPOINT or '未设置'}")
print(f"• 参考音频路径: {REFERENCE_AUDIO_PATH or '未设置'}")
print()

# 示例配置（用户需要取消注释并填入实际值）
# COMFYUI_SERVER_URL = "your-comfyui-server.com:8080"
# COMFYUI_WORKFLOW_PATH = "path/to/your/workflow.json"
# GPT_SOVITS_ENDPOINT = "your-gpt-sovits-endpoint-name"
# REFERENCE_AUDIO_PATH = "s3://your-bucket/reference-audio.mp3"

print("📁 请将漫画图像放入 'input_images' 目录")
print("然后运行下面的代码开始处理")

📋 配置检查
请确保以下配置已正确设置：
• ComfyUI服务器: http://ec2-35-84-2-12.us-west-2.compute.amazonaws.com:8188
• ComfyUI工作流: ./sample_workflow.json
• GPT-SoVITS端点: gpt-sovits-inference-2025-08-12-07-51-09-161
• 参考音频路径: s3://sagemaker-us-west-2-687912291502/gpt-sovits/wav/speech_20240425104005663.mp3

📁 请将漫画图像放入 'input_images' 目录
然后运行下面的代码开始处理


### 完整流程处理

运行完整的漫画转视频配音流程（包含优化步骤）：

In [44]:
# 运行完整流程
# 注意：请确保已正确配置所有参数

input_dir = "input_images"

# 检查输入目录
if not os.path.exists(input_dir):
    print(f"❌ 输入目录不存在: {input_dir}")
    print("请创建目录并放入漫画图像")
else:
    images = get_comic_images_from_directory(input_dir)
    if not images:
        print(f"❌ 输入目录中没有图像文件: {input_dir}")
        print("请放入 .jpg, .jpeg, .png, .bmp, .gif 格式的图像")
    else:
        print(f"✅ 找到 {len(images)} 个图像文件")
        
        # 检查配置
        config_ok = True
        if not COMFYUI_SERVER_URL:
            print("❌ 请设置 COMFYUI_SERVER_URL")
            config_ok = False
        if not COMFYUI_WORKFLOW_PATH:
            print("❌ 请设置 COMFYUI_WORKFLOW_PATH")
            config_ok = False
        if not GPT_SOVITS_ENDPOINT:
            print("❌ 请设置 GPT_SOVITS_ENDPOINT")
            config_ok = False
        if not REFERENCE_AUDIO_PATH:
            print("❌ 请设置 REFERENCE_AUDIO_PATH")
            config_ok = False
        
        if config_ok:
            print("\n🚀 开始处理...")
            final_video = process_comic_to_video_voice(input_dir)
            
            if final_video:
                print(f"\n🎉 处理完成！")
                print(f"📹 最终视频: {final_video}")
            else:
                print("\n❌ 处理失败")
        else:
            print("\n⚠️ 请先完成配置再运行")

📂 扫描目录: input_images
  ✅ 找到图像: page_003.png
  ✅ 找到图像: page_011.png
  ✅ 找到图像: page_009.png
  ✅ 找到图像: page_002.png
  ✅ 找到图像: page_008.png
  ✅ 找到图像: page_005.png
  ✅ 找到图像: page_004.png
  ✅ 找到图像: page_006.png
  ✅ 找到图像: page_012.png
  ✅ 找到图像: page_010.png
  ✅ 找到图像: page_001.png
  ✅ 找到图像: page_007.png
📊 总共找到 12 个图像文件
✅ 找到 12 个图像文件

🚀 开始处理...
🚀 开始漫画转视频配音流程

📊 步骤1: 分析漫画图像内容
📂 扫描目录: input_images
  ✅ 找到图像: page_003.png
  ✅ 找到图像: page_011.png
  ✅ 找到图像: page_009.png
  ✅ 找到图像: page_002.png
  ✅ 找到图像: page_008.png
  ✅ 找到图像: page_005.png
  ✅ 找到图像: page_004.png
  ✅ 找到图像: page_006.png
  ✅ 找到图像: page_012.png
  ✅ 找到图像: page_010.png
  ✅ 找到图像: page_001.png
  ✅ 找到图像: page_007.png
📊 总共找到 12 个图像文件

🚀 开始批量分析 12 个图像（多图像模式）...

📦 处理批次 1: 图像 1-6
📁 批次包含图像:
  1. page_001.png
  2. page_002.png
  3. page_003.png
  4. page_004.png
  5. page_005.png
  6. page_006.png
  ✅ 批次分析完成
  🔑 选中图像: 第6张 - page_006.png
  🎭 主要主题: 主要主题是角色在一个充满魔法和神秘生物的世界中经历冒险和挑战。情节发展包括角色遇到危险、展现特殊能力、面临追逐和经历情感波动。
  🎬 视频提示: 将第六张图像动起来，展示角色在悲伤的表情下，泪水从眼中流下，同

MoviePy - Done.
MoviePy - Writing video final_videos/final_000.mp4



MoviePy - Done !
MoviePy - video ready final_videos/final_000.mp4
✅ 字幕视频已保存: final_videos/final_000.mp4
💾 保留临时文件用于调试: temp/temp_merged_d540ca6a.mp4

🎬 合并视频音频并添加字幕 2/2
🎞️ 合并视频和音频
📹 视频: video_001.mp4
🔊 音频: audio_001.wav
✅ 合并完成: temp/temp_merged_a529787f.mp4
📝 添加字幕到视频
📹 输入视频: temp_merged_a529787f.mp4
💬 字幕内容: 在一个充满魔法的世界，女孩在森林中穿梭逃跑，背后狼群紧追不舍，她充满恐惧和决心。
MoviePy - Building video final_videos/final_001.mp4.
MoviePy - Writing audio in final_001TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video final_videos/final_001.mp4



frame_index:  88%|████████▊ | 109/124 [00:01<00:00, 64.81it/s, now=None]/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file temp/temp_merged_a529787f.mp4, 1105920 bytes wanted but 0 bytes read at frame index 113 (out of a total 124 frames), at time 7.06/7.80 sec. Using the last valid frame instead.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file temp/temp_merged_a529787f.mp4, 1105920 bytes wanted but 0 bytes read at frame index 114 (out of a total 124 frames), at time 7.12/7.80 sec. Using the last valid frame instead.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/moviepy/video/io/ffmpeg_reader.py:190: UserWarning: In file temp/temp_merged_a529787f.mp4, 1105920 bytes wanted but 0 bytes read at frame index 115 (out of a total 124 frames), at time 7.19/7.80 sec. Using 

MoviePy - Done !
MoviePy - video ready final_videos/final_001.mp4
✅ 字幕视频已保存: final_videos/final_001.mp4
💾 保留临时文件用于调试: temp/temp_merged_a529787f.mp4

🎬 步骤6: 拼接最终视频
🎬 拼接 2 个视频文件
  1. final_000.mp4
  2. final_001.mp4
✅ 视频拼接完成: final_videos/comic_video_20250814_071900.mp4

🎉 流程完成！
📁 最终视频: final_videos/comic_video_20250814_071900.mp4
⏱️ 总时长: 14.89 秒
📊 处理统计:
  • 输入图像: 2
  • 生成视频: 2
  • 生成语音: 2
  • 最终视频位置: final_videos/ 目录

🎉 处理完成！
📹 最终视频: final_videos/comic_video_20250814_071900.mp4


In [ ]:
# 单独运行提示词和脚本优化
# 这将读取 temp/selected_images_info.json 文件并优化其中的内容

print("🔧 开始优化提示词和配音脚本...")
success = optimize_prompts_and_scripts_with_claude()

if success:
    print("\n✅ 优化完成！")
    print("📁 优化后的数据已保存到 temp/selected_images_info.json")
    print("\n💡 优化内容包括:")
    print("  • video_prompt: 优化为符合ComfyUI的英文提示词")
    print("  • combined_audio_script: 优化为连贯的中文配音脚本")
    print("  • 确保人物和主体的一致性")
else:
    print("\n❌ 优化失败，请检查配置和网络连接")

## 10. 工具函数和调试

In [ ]:
# 单独测试Bedrock图像分析
def test_image_analysis(image_path: str):
    """
    测试单张图像的分析功能
    """
    if not os.path.exists(image_path):
        print(f"❌ 图像文件不存在: {image_path}")
        return
    
    print(f"🔍 测试图像分析: {image_path}")
    result = analyze_comic_image_content(image_path)
    
    if result.get('success'):
        analysis = result['analysis_result']
        print("\n✅ 分析结果:")
        for key, value in analysis.items():
            print(f"  {key}: {value}")
    else:
        print(f"❌ 分析失败: {result.get('error')}")

# 测试简化的批量图像分析
def test_batch_analysis_simplified(image_dir: str, max_images: int = 5):
    """
    测试简化后的批量图像分析功能
    """
    if not os.path.exists(image_dir):
        print(f"❌ 目录不存在: {image_dir}")
        return
    
    print(f"🔍 测试简化批量分析: {image_dir}")
    print(f"📊 最大处理图像数: {max_images}")
    
    # 获取图像文件
    image_files = get_comic_images_from_directory(image_dir)
    if not image_files:
        print("❌ 目录中没有找到图像文件")
        return
    
    # 限制处理数量
    test_files = image_files[:max_images]
    print(f"\n📁 测试文件列表:")
    for i, file_path in enumerate(test_files, 1):
        print(f"  {i}. {os.path.basename(file_path)}")
    
    # 调用批量分析
    result = analyze_comic_images_batch(test_files, 0)
    
    if result.get('success'):
        analysis = result['analysis_result']
        print("\n✅ 简化批量分析结果:")
        
        # 显示整体分析
        if 'overall_analysis' in analysis:
            print("\n📊 整体分析:")
            overall = analysis['overall_analysis']
            for key, value in overall.items():
                print(f"  {key}: {value}")
        
        # 显示选中的图像索引
        if 'selected_index' in analysis:
            selected_idx = analysis['selected_index']
            if 0 <= selected_idx < len(test_files):
                selected_file = test_files[selected_idx]
                print(f"\n🎯 选中图像: 第{selected_idx + 1}张 - {os.path.basename(selected_file)}")
            else:
                print(f"\n🎯 选中图像索引: {selected_idx} (索引超出范围)")
        
        # 显示视频提示词
        if 'video_prompt' in analysis:
            video_prompt = analysis['video_prompt']
            print(f"\n🎬 视频提示词: {video_prompt}")
        
        # 显示配音文本
        if 'combined_audio_script' in analysis:
            audio_script = analysis['combined_audio_script']
            print(f"\n🎙️ 配音文本: {audio_script}")
        
        print(f"\n⏱️ 处理耗时: {result.get('latency', {}).get('total_ms', 0):.2f}ms")
        
    else:
        print(f"❌ 批量分析失败: {result.get('error')}")

# 清理临时文件
def cleanup_temp_files():
    """
    清理临时文件和目录
    """
    import shutil
    
    temp_dirs = ['temp', 'output_videos', 'output_audio']
    for temp_dir in temp_dirs:
        if os.path.exists(temp_dir):
            try:
                shutil.rmtree(temp_dir)
                os.makedirs(temp_dir, exist_ok=True)
                print(f"🧹 已清理: {temp_dir}")
            except Exception as e:
                print(f"❌ 清理失败 {temp_dir}: {e}")

# 检查依赖
def check_dependencies():
    """
    检查系统依赖
    """
    print("🔧 检查系统依赖:")
    
    # 检查ffmpeg
    try:
        result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        if result.returncode == 0:
            print("  ✅ ffmpeg 已安装")
        else:
            print("  ❌ ffmpeg 未正确安装")
    except FileNotFoundError:
        print("  ❌ ffmpeg 未找到")
    
    # 检查ffprobe
    try:
        result = subprocess.run(['ffprobe', '-version'], capture_output=True, text=True)
        if result.returncode == 0:
            print("  ✅ ffprobe 已安装")
        else:
            print("  ❌ ffprobe 未正确安装")
    except FileNotFoundError:
        print("  ❌ ffprobe 未找到")
    
    # 检查Python包
    required_packages = ['boto3', 'requests', 'pydub']
    for package in required_packages:
        try:
            __import__(package)
            print(f"  ✅ {package} 已安装")
        except ImportError:
            print(f"  ❌ {package} 未安装")

print("✅ 工具函数已定义")
print("\n🎯 使用说明:")
print("1. 运行 check_dependencies() 检查系统依赖")
print("2. 配置 ComfyUI 和 GPT-SoVITS 参数")
print("3. 将漫画图像放入 input_images 目录")
print("4. 运行主流程开始处理")
print("5. 使用 test_image_analysis() 测试单张图像")
print("6. 使用 test_batch_analysis_simplified() 测试简化批量分析")
print("7. 使用 cleanup_temp_files() 清理临时文件")

## 11. 测试简化的批量分析功能

In [ ]:
# 测试简化后的批量图像分析功能
print("🧪 测试简化的批量图像分析功能")
print("="*50)

# 检查输入目录
input_dir = 'input_images'
if os.path.exists(input_dir):
    # 测试少量图像的批量分析
    print(f"\n📂 测试目录: {input_dir}")
    print("🔍 运行简化批量分析（最多3张图像）...")
    
    # 调用测试函数
    test_batch_analysis_simplified(input_dir, max_images=3)
    
    print("\n" + "="*50)
    print("📋 简化输出说明:")
    print("• overall_analysis: 整体故事分析")
    print("• selected_index: 选中的图像序号（从0开始）")
    print("• video_prompt: 完整视频生成提示词")
    print("• combined_audio_script: 完整配音文本")
    print("\n✨ 不再输出每张图像的详细分析，输出更简洁！")
    
else:
    print(f"❌ 输入目录不存在: {input_dir}")
    print("请先创建目录并放入一些漫画图像文件")

## 使用示例

In [ ]:
# 示例1: 使用ComfyUI模式处理漫画
if __name__ == "__main__":
    # 设置使用ComfyUI
    USE_COMFYUI = True
    
    # 处理漫画目录
    input_dir = "input_images"
    if os.path.exists(input_dir) and os.listdir(input_dir):
        print("🎬 开始ComfyUI模式处理...")
        final_video = process_comic_to_video_voice(input_dir)
        if final_video:
            print(f"✅ 处理完成: {final_video}")
    else:
        print("⚠️ 请先将漫画图像放入 input_images 目录")

In [ ]:
# 示例2: 使用MoviePy模式处理漫画
if __name__ == "__main__":
    # 设置使用MoviePy
    USE_COMFYUI = False
    MOVIEPY_DURATION = 5.0
    MOVIEPY_EFFECT = "random"
    
    # 处理漫画目录
    input_dir = "input_images"
    if os.path.exists(input_dir) and os.listdir(input_dir):
        print("🎬 开始MoviePy模式处理...")
        final_video = process_comic_to_video_voice(input_dir)
        if final_video:
            print(f"✅ 处理完成: {final_video}")
    else:
        print("⚠️ 请先将漫画图像放入 input_images 目录")

In [ ]:
# 示例3: 使用已有的selected_images_info.json文件通过MoviePy生成视频
if __name__ == "__main__":
    selected_file = "temp/selected_images_info.json"
    if os.path.exists(selected_file):
        print("🎬 使用MoviePy处理已选中的图像...")
        final_video = process_selected_images_with_moviepy(selected_file)
        if final_video:
            print(f"✅ 处理完成: {final_video}")
    else:
        print("⚠️ 未找到selected_images_info.json文件，请先运行图像分析步骤")

In [ ]:
# 示例4: 测试单个图像的MoviePy动画效果
if __name__ == "__main__":
    test_image = "input_images/page_001.png"  # 替换为实际的图像路径
    if os.path.exists(test_image):
        print("🎬 测试MoviePy动画效果...")
        
        # 测试不同效果
        effects = ["shake_zoom", "fade_zoom", "pan_zoom", "random"]
        
        for effect in effects:
            output_path = f"output_videos/test_{effect}.mp4"
            print(f"\n🎭 测试效果: {effect}")
            
            success = generate_video_from_image_moviepy(
                image_path=test_image,
                video_prompt="Test animation effect",
                output_path=output_path,
                duration=3.0,
                effect=effect
            )
            
            if success:
                print(f"✅ {effect} 效果测试成功: {output_path}")
            else:
                print(f"❌ {effect} 效果测试失败")
    else:
        print("⚠️ 测试图像不存在，请检查路径")